In [3]:
pip install pyspark

In [4]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator


In [5]:
spark = SparkSession.builder \
    .appName("LinearRegressionExample") \
    .getOrCreate()

In [7]:
df = spark.read.csv("/content/BostonHousing.csv", header=True, inferSchema=True)

df.show()
df.printSchema()

+-------+----+-----+----+-----+-----+-----+------+---+---+-------+------+-----+----+
|   crim|  zn|indus|chas|  nox|   rm|  age|   dis|rad|tax|ptratio|     b|lstat|medv|
+-------+----+-----+----+-----+-----+-----+------+---+---+-------+------+-----+----+
|0.00632|18.0| 2.31|   0|0.538|6.575| 65.2|  4.09|  1|296|   15.3| 396.9| 4.98|24.0|
|0.02731| 0.0| 7.07|   0|0.469|6.421| 78.9|4.9671|  2|242|   17.8| 396.9| 9.14|21.6|
|0.02729| 0.0| 7.07|   0|0.469|7.185| 61.1|4.9671|  2|242|   17.8|392.83| 4.03|34.7|
|0.03237| 0.0| 2.18|   0|0.458|6.998| 45.8|6.0622|  3|222|   18.7|394.63| 2.94|33.4|
|0.06905| 0.0| 2.18|   0|0.458|7.147| 54.2|6.0622|  3|222|   18.7| 396.9| 5.33|36.2|
|0.02985| 0.0| 2.18|   0|0.458| 6.43| 58.7|6.0622|  3|222|   18.7|394.12| 5.21|28.7|
|0.08829|12.5| 7.87|   0|0.524|6.012| 66.6|5.5605|  5|311|   15.2| 395.6|12.43|22.9|
|0.14455|12.5| 7.87|   0|0.524|6.172| 96.1|5.9505|  5|311|   15.2| 396.9|19.15|27.1|
|0.21124|12.5| 7.87|   0|0.524|5.631|100.0|6.0821|  5|311|   15.2

In [8]:
feature_cols = [
    "crim", "zn", "indus", "chas", "nox", "rm",
    "age", "dis", "rad", "tax", "ptratio", "b", "lstat"]

In [9]:
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

In [10]:
assembler = VectorAssembler( inputCols=feature_cols, outputCol="features")


In [12]:
data = assembler.transform(df)
final_data = data.select("features", "medv") \
                 .withColumnRenamed("medv", "label")


In [13]:
train, test = final_data.randomSplit([0.8, 0.2], seed=42)

In [14]:
lr = LinearRegression(
    featuresCol="features",
    labelCol="label")

In [15]:
model = lr.fit(train)
predictions = model.transform(test)
predictions.select("label", "prediction").show(10)

+-----+------------------+
|label|        prediction|
+-----+------------------+
| 22.0|27.482274018186136|
| 50.0| 40.59821928572501|
| 29.1|31.560171030407385|
| 32.9| 30.50410754091398|
| 42.3| 36.71084264945597|
| 34.7|30.375442767948023|
| 18.5| 19.59899004373776|
| 34.9|34.091019647336594|
| 23.5|30.499298725303998|
| 27.9| 31.92001039736025|
+-----+------------------+
only showing top 10 rows


In [17]:
evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction"
)

rmse = evaluator.setMetricName("rmse").evaluate(predictions)
mse = evaluator.setMetricName("mse").evaluate(predictions)
mae = evaluator.setMetricName("mae").evaluate(predictions)
r2 = evaluator.setMetricName("r2").evaluate(predictions)

print(f"RMSE = {rmse:.4f}")
print(f"MSE  = {mse:.4f}")
print(f"MAE  = {mae:.4f}")
print(f"R²   = {r2:.4f}")
#parameters
print("Coefficients:", model.coefficients)
print("Intercept:", model.intercept)

RMSE = 4.6718
MSE  = 21.8258
MAE  = 3.5081
R²   = 0.7932
Coefficients: [-0.11362203729409111,0.048909186934055354,0.023795428986748923,2.801771998735071,-18.415424541191168,3.5158797633118337,0.005211682161471176,-1.4163830723540525,0.3317669315937271,-0.013607893704165114,-0.953414333840846,0.008602677392852453,-0.5195035312476768]
Intercept: 38.616991445737995
